In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Link Prediction on Directed Graphs using Node2Vec (Biased Random Walks)
=======================================================================
This script builds a directed graph from an XML file (e.g., oncogene_pathways.xml),
generates biased random walks respecting edge directions, learns node embeddings
with Word2Vec, trains a Random Forest classifier on edge embeddings (concatenated
source and target vectors), predicts missing directed edges, and validates
predictions against the STRING database.

Usage:
    python 07_link_prediction_node2vec_directed.py
"""

import os
import random
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

import numpy as np
import pandas as pd
import networkx as nx
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import requests

# Set random seeds for reproducibility
random.seed(12345)
np.random.seed(12345)

# ----------------------------------------------------------------------
# 1. Build directed graph from XML
# ----------------------------------------------------------------------
def build_graph_from_xml(xml_file):
    """Parse XML and return a directed graph."""
    G = nx.DiGraph()
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    G.add_node(node_name)
        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode').get('node')
                end = edge.find('endNode').get('node')
                if start and end:
                    G.add_edge(start, end)
    return G

# ----------------------------------------------------------------------
# 2. Biased random walk for directed graphs (Node2Vec adaptation)
# ----------------------------------------------------------------------
def biased_random_walk(G, start_node, walk_length, p, q):
    """
    Generate a single biased random walk on a directed graph.
    Transition probabilities depend on the previous node to simulate
    second‑order random walks.
    """
    walk = [start_node]
    while len(walk) < walk_length:
        cur = walk[-1]
        out_neighbors = list(G.successors(cur))
        if not out_neighbors:
            break
        if len(walk) == 1:
            next_node = random.choice(out_neighbors)
            walk.append(next_node)
            continue

        prev = walk[-2]
        probs = []
        for neighbor in out_neighbors:
            if neighbor == prev:
                probs.append(1 / p)
            elif G.has_edge(neighbor, prev):
                probs.append(1)
            else:
                probs.append(1 / q)
        probs = np.array(probs) / np.sum(probs)
        next_node = np.random.choice(out_neighbors, p=probs)
        walk.append(next_node)
    return walk

# ----------------------------------------------------------------------
# 3. Generate walks and train Word2Vec
# ----------------------------------------------------------------------
def generate_walks(G, num_walks, walk_length, p, q):
    """Generate random walks for all nodes."""
    nodes = list(G.nodes())
    walks = []
    for node in nodes:
        for _ in range(num_walks):
            walk = biased_random_walk(G, node, walk_length, p, q)
            walks.append([str(x) for x in walk])
    return walks

def train_word2vec(walks, dimensions, window, workers=4):
    """Train Word2Vec model on walks."""
    model = Word2Vec(sentences=walks, vector_size=dimensions, window=window,
                     min_count=1, sg=1, workers=workers, seed=42)
    return model

# ----------------------------------------------------------------------
# 4. Negative sampling with degree matching (using total degree)
# ----------------------------------------------------------------------
def degree_matching_negative_sampling(positive_edges, G, nodes):
    """Sample negative edges matching degree distribution of positive edges."""
    degree = dict(G.degree())
    degree_vals = list(degree.values())
    degree_bins = np.percentile(degree_vals, [25, 50, 75])

    def degree_bucket(d):
        if d <= degree_bins[0]:
            return 0
        elif d <= degree_bins[1]:
            return 1
        elif d <= degree_bins[2]:
            return 2
        else:
            return 3

    # Bucket distribution of positive edges
    pos_bucket_dist = defaultdict(int)
    for u, v in positive_edges:
        key = (degree_bucket(degree[u]), degree_bucket(degree[v]))
        pos_bucket_dist[key] += 1

    # Nodes per bucket
    bucket_to_nodes = defaultdict(list)
    for n, d in degree.items():
        bucket_to_nodes[degree_bucket(d)].append(n)

    existing_edges_set = set(positive_edges)
    negative_edges = []

    for (bu, bv), count in pos_bucket_dist.items():
        candidates_u = bucket_to_nodes[bu]
        candidates_v = bucket_to_nodes[bv]
        sampled = 0
        attempts = 0
        while sampled < count and attempts < count * 20:
            u = random.choice(candidates_u)
            v = random.choice(candidates_v)
            if u != v and (u, v) not in existing_edges_set:
                negative_edges.append((u, v))
                sampled += 1
            attempts += 1
        # Fill remaining with fully random sampling if needed (rare)
        while sampled < count:
            u = random.choice(nodes)
            v = random.choice(nodes)
            if u != v and (u, v) not in existing_edges_set:
                negative_edges.append((u, v))
                sampled += 1
    return negative_edges

# ----------------------------------------------------------------------
# 5. Edge feature: concatenation of source and target embeddings
# ----------------------------------------------------------------------
def edge_features(embeddings, u, v):
    """Concatenate embeddings of source and target nodes."""
    return np.concatenate([embeddings[u], embeddings[v]])

# ----------------------------------------------------------------------
# 6. STRING validation helper
# ----------------------------------------------------------------------
def check_string_interaction(protein1, protein2, species=9606, timeout=10):
    """Query STRING API to check if an interaction is known."""
    base_url = "https://string-db.org/api/json/network"
    params = {
        "identifiers": f"{protein1}\n{protein2}",
        "species": species,
        "required_score": 0,
        "caller_identity": "my_prediction"
    }
    try:
        resp = requests.get(base_url, params=params, timeout=timeout)
        if resp.status_code != 200:
            return False, 0.0
        data = resp.json()
        for edge in data:
            if (edge['preferredName_A'] == protein1 and edge['preferredName_B'] == protein2) or \
               (edge['preferredName_A'] == protein2 and edge['preferredName_B'] == protein1):
                score = float(edge.get('combined_score', 0))
                return True, score
        return False, 0.0
    except Exception as e:
        print(f"API error for {protein1}-{protein2}: {e}")
        return False, 0.0

# ----------------------------------------------------------------------
# 7. Main pipeline
# ----------------------------------------------------------------------
def main():
    # Input file
    xml_file = "oncogene_pathways.xml"
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"Input file {xml_file} not found. Please run previous steps first.")

    # Build graph
    G = build_graph_from_xml(xml_file)
    print(f"Directed graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Random walk parameters
    num_walks = 200
    walk_length = 30
    p = 1.0
    q = 1.0
    dimensions = 64
    window = 10

    # Generate walks
    print("Generating biased random walks...")
    walks = generate_walks(G, num_walks, walk_length, p, q)
    print(f"Generated {len(walks)} walks")

    # Train Word2Vec
    print("Training Word2Vec model...")
    model_w2v = train_word2vec(walks, dimensions, window)
    embeddings = {node: model_w2v.wv[str(node)] for node in G.nodes()}
    print(f"Embedding dimension: {len(list(embeddings.values())[0])}")

    # Positive edges
    positive_edges = list(G.edges())
    print(f"Positive samples: {len(positive_edges)}")

    # Negative sampling with degree matching
    nodes = list(G.nodes())
    negative_edges = degree_matching_negative_sampling(positive_edges, G, nodes)
    print(f"Negative samples: {len(negative_edges)}")

    # Build feature matrix
    X_pos = np.array([edge_features(embeddings, u, v) for u, v in positive_edges])
    X_neg = np.array([edge_features(embeddings, u, v) for u, v in negative_edges])
    X = np.vstack([X_pos, X_neg])
    y = np.hstack([np.ones(len(positive_edges)), np.zeros(len(negative_edges))])
    print(f"Feature matrix shape: {X.shape}")

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Random Forest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)

    # Evaluation
    y_pred_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)
    print(f"Test AUC: {auc:.4f}")
    print(classification_report(y_test, clf.predict(X_test)))

    # Predict all missing directed edges
    existing_edges_set = set(positive_edges)
    all_pairs = [(u, v) for u in nodes for v in nodes if u != v and (u, v) not in existing_edges_set]
    print(f"Total directed non‑edge pairs to predict: {len(all_pairs)}")

    batch_size = 10000
    pred_results = []
    for i in range(0, len(all_pairs), batch_size):
        batch = all_pairs[i:i+batch_size]
        X_batch = np.array([edge_features(embeddings, u, v) for u, v in batch])
        probs = clf.predict_proba(X_batch)[:, 1]
        for (u, v), p in zip(batch, probs):
            pred_results.append((u, v, p))
        if (i + batch_size) % 50000 == 0:
            print(f"Predicted {i+batch_size} pairs")
    print()

    pred_df = pd.DataFrame(pred_results, columns=['source', 'target', 'probability'])
    pred_df = pred_df.sort_values('probability', ascending=False).reset_index(drop=True)

    print("Top 20 predicted directed edges:")
    print(pred_df.head(20))

    # Save top predictions
    top_n = 1000
    output_file = "directed_link_predictions_manual.csv"
    pred_df.head(top_n).to_csv(output_file, index=False)
    print(f"Saved top {top_n} predictions to {output_file}")

    # Optional: export graph with high‑confidence predictions
    high_conf = pred_df[pred_df['probability'] >= 0.95]
    G_new = G.copy()
    for _, row in high_conf.iterrows():
        G_new.add_edge(row['source'], row['target'], predicted=True, probability=row['probability'])
    nx.write_graphml(G_new, "directed_network_with_predictions.graphml")
    print(f"Exported graph with {G_new.number_of_edges()} edges (including {len(high_conf)} predicted)")

    # STRING validation on top 100 predictions
    print("\nValidating top 100 predictions with STRING...")
    top100 = pred_df.head(100).copy()
    validated = []
    for idx, row in top100.iterrows():
        u, v = row['source'], row['target']
        exists, score = check_string_interaction(u, v)
        validated.append({
            'source': u,
            'target': v,
            'probability': row['probability'],
            'string_exists': exists,
            'string_score': score
        })
        time.sleep(0.5)  # rate limiting

    val_df = pd.DataFrame(validated)
    val_df.to_csv("string_validation_top100.csv", index=False)
    correct = val_df['string_exists'].sum()
    precision = correct / 100
    print(f"\n=== STRING Validation (k=100) ===")
    print(f"Edges found in STRING: {correct}")
    print(f"Precision@100: {precision:.4f}")
    print("Detailed validation saved to string_validation_top100.csv")

if __name__ == "__main__":
    main()